<a href="https://colab.research.google.com/github/RIVALX-1/sae_dlm_cross_entropy/blob/main/sae_dlm_cross_entropy_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/RIVALX-1/sae_dlm_cross_entropy
%cd sae_dlm_cross_entropy

In [ ]:
!pip install transformers==4.46.2
!pip install torch
!pip install git+https://github.com/saprmarks/dictionary_learning.git
!pip install huggingface_hub

In [ ]:
from dictionary_learning import AutoEncoder, utils
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download
import torch
from huggingface_hub import login
login()

In [ ]:
from dictionary_learning.trainers.top_k import AutoEncoderTopK

class TopKSae(AutoEncoderTopK):

    @classmethod
    def from_ae_pt(cls, repo_id: str, layer: int) -> "TopKSae":
        path = hf_hub_download(
            repo_id=repo_id,
            filename=f"saes_mask_Dream-org_Dream-v0-Base-7B_top_k/"
                     f"resid_post_layer_{layer}/trainer_0/ae.pt"
        )
        ckpt = torch.load(path, map_location="cpu")

        dict_size, activation_dim = ckpt["encoder.weight"].shape
        k = int(ckpt["k"])

        sae = cls(activation_dim, dict_size, k)

        # let AutoEncoderTopK load its own keys as-is
        sae.load_state_dict(ckpt, strict=False)
        return sae

sae = TopKSae.from_ae_pt("AwesomeInterpretability/dlm-mask-topk-sae", layer=5)
print(sae)
print(sae.b_dec)

In [ ]:
print('We have access to a GPU:', torch.cuda.is_available())
!nvidia-smi

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("AwesomeInterpretability/dlm-mask-topk-sae")

for f in files:
    print(f)

In [ ]:
layer = 5

ae_path = hf_hub_download(
    repo_id="AwesomeInterpretability/dlm-mask-topk-sae",
    filename=f"saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_{layer}/trainer_0/ae.pt"
)

In [ ]:
#downloading all the layer files at once

#files = list_repo_files("AwesomeInterpretability/dlm-mask-topk-sae")

#ae_files = [f for f in files if f.endswith("ae.pt")]

#for f in ae_files:
#    path = hf_hub_download(
#        repo_id="AwesomeInterpretability/dlm-mask-topk-sae",
#        filename=f
#    )

In [ ]:
ae = torch.load(ae_path)
dict_size, activation_dim = ae["encoder.weight"].shape

autoencoder = AutoEncoder(activation_dim, dict_size)

ae['bias'] = ae['b_dec']
del ae['b_dec']
del ae['k']
del ae['threshold']

autoencoder.load_state_dict(ae)

In [ ]:
model = AutoModel.from_pretrained(
    "Dream-org/Dream-v0-Base-7B",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

In [ ]:
model = model.cuda()
model.eval()

In [ ]:
def splice_sae(module, input, output):
    with torch.no_grad():
        encoded = autoencoder.encode(output)
        reconstructed = autoencoder.decode(encoded)
    return reconstructed


In [ ]:
model_path = "GSAI-ML/LLaDA-8B-Base"
messages = [
    {"role": "user", "content": "Please write a Python class that implements a PyTorch trainer capable of training a model on a toy dataset."}
]
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
)
input_ids = inputs.input_ids.to(device="cuda")
attention_mask = inputs.attention_mask.to(device="cuda")

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

model_path = "GSAI-ML/LLaDA-8B-Base"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)

model.eval()

In [ ]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask
    )

print(output.logits.shape)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
def geometric_pmf(p: float, k: torch.Tensor) -> torch.Tensor:
    return p * torch.pow(1.0 - p, k)

def sample_exact_k_mask(maskable_positions: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    B, L = maskable_positions.shape
    device = maskable_positions.device
    mask = torch.zeros_like(maskable_positions, dtype=torch.bool)

    for b in range(B):
        valid_idx = torch.where(maskable_positions[b])[0]
        m = valid_idx.numel()
        if m == 0:
            continue

        k = int(torch.round(t[b] * m).item())
        k = max(1, min(k, m))

        perm = torch.randperm(m, device=device)
        chosen = valid_idx[perm[:k]]
        mask[b, chosen] = True

    return mask

In [ ]:
def compute_dlm_cross_entropy(
    model,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    mask_token_id: int,
    loss_mask: torch.Tensor | None = None,
    mode: str = "llada",
    exact_k_masking: bool = True,
    cart_p: float = 0.5,
    position_ids: torch.Tensor | None = None,
):
    device = input_ids.device
    B, L = input_ids.shape

    if loss_mask is None:
        maskable_positions = attention_mask.bool()
    else:
        maskable_positions = attention_mask.bool() & loss_mask.bool()

    t = torch.rand(B, device=device).clamp_min(1e-4)

    if exact_k_masking:
        mask_positions = sample_exact_k_mask(maskable_positions, t)
    else:
        rand = torch.rand(B, L, device=device)
        mask_positions = (rand < t[:, None]) & maskable_positions
        for b in range(B):
            if maskable_positions[b].any() and not mask_positions[b].any():
                first_valid = torch.where(maskable_positions[b])[0][0]
                mask_positions[b, first_valid] = True

    corrupted_input_ids = input_ids.clone()
    corrupted_input_ids[mask_positions] = mask_token_id

    forward_kwargs = {
        "input_ids": corrupted_input_ids,
        "attention_mask": attention_mask,
    }
    if position_ids is not None:
        forward_kwargs["position_ids"] = position_ids

    outputs = model(**forward_kwargs)
    logits = outputs.logits

    ce_fct = nn.CrossEntropyLoss(reduction="none")
    per_token_ce = ce_fct(
        logits.view(-1, logits.size(-1)),
        input_ids.view(-1)
    ).view(B, L)

    weight_map = mask_positions.float()

    if mode in ("llada", "dream_base"):
        weight_map = weight_map * (1.0 / t)[:, None]

    elif mode == "dream_cart":
        visible_positions = (~mask_positions) & maskable_positions
        idx = torch.arange(L, device=device)
        cart_weights = torch.zeros(B, L, device=device)

        for b in range(B):
            visible_idx = torch.where(visible_positions[b])[0]
            if visible_idx.numel() == 0:
                cart_weights[b] = 1.0
                continue

            n_idx = idx[:, None]
            i_idx = visible_idx[None, :]
            k = (n_idx - i_idx).abs() - 1
            k = torch.clamp(k, min=0)

            geo_vals = geometric_pmf(cart_p, k.float())
            cart_weights[b] = 0.5 * geo_vals.sum(dim=1)

        weight_map = weight_map * (1.0 / t)[:, None] * cart_weights

    else:
        raise ValueError("mode must be one of: 'llada', 'dream_base', 'dream_cart'")

    weighted_loss_sum = (per_token_ce * weight_map).sum()
    normalizer = weight_map.sum().clamp_min(1e-8)
    loss = weighted_loss_sum / normalizer

    return {
        "loss": loss,
        "t": t,
        "corrupted_input_ids": corrupted_input_ids,
        "mask_positions": mask_positions,
        "logits": logits,
        "per_token_ce": per_token_ce,
        "weight_map": weight_map,
    }

In [ ]:
prompt = "The capital of France is Paris."

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids.to(model.device)
attention_mask = inputs.attention_mask.to(model.device)

dlm_mask_token_id = tokenizer.mask_token_id
if dlm_mask_token_id is None:
    if tokenizer.eos_token_id is not None:
        dlm_mask_token_id = tokenizer.eos_token_id
        print(f"Warning: tokenizer.mask_token_id is None. Using tokenizer.eos_token_id ({dlm_mask_token_id}) as mask_token_id.")
    elif tokenizer.pad_token_id is not None:
        dlm_mask_token_id = tokenizer.pad_token_id
        print(f"Warning: tokenizer.mask_token_id is None and tokenizer.eos_token_id is None. Using tokenizer.pad_token_id ({dlm_mask_token_id}) as mask_token_id.")
    else:
        # Fallback to an arbitrary token ID if no suitable special token is found.
        # This is a risky fallback and might not be semantically correct for the DLM task.
        dlm_mask_token_id = 0 # Assuming 0 is a valid token ID in the vocabulary
        print(f"Warning: No suitable mask, eos, or pad token found. Using token ID 0 as mask_token_id. This might lead to unexpected behavior.")

print("mask_token_id:", dlm_mask_token_id)

result = compute_dlm_cross_entropy(
    model=model,
    input_ids=input_ids,
    attention_mask=attention_mask,
    mask_token_id=dlm_mask_token_id,
    mode="llada",
    exact_k_masking=True
)

print("Loss:", result["loss"].item())

In [ ]:
!nvidia-smi

In [ ]:
from datasets import load_dataset

ds = load_dataset("common-pile/wikiteam")

In [ ]:
!pip install transformers datasets accelerate

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "common-pile/wikiteam",
    split="train",
    streaming=True
)

In [ ]:
from itertools import islice

subset = list(islice(ds, 10))

In [ ]:
text = subset[0]["text"]
print(text[:500])

In [ ]:
!pip install bitsandbytes==0.43.2
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import torch

# Clear memory from any previous models
if 'model' in globals() and model is not None:
    del model
    torch.cuda.empty_cache()

model_path = "GSAI-ML/LLaDA-8B-Base"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModel.from_pretrained(
    model_path,
    quantization_config=quant_config,
    trust_remote_code=True,
    device_map="auto"
)

model.eval()

ds = load_dataset("common-pile/wikiteam", split="train", streaming=True)

text = next(iter(ds))["text"]

inputs = tokenizer(text[:100], return_tensors="pt").to(model.device)
outputs = model(**inputs)